In [1]:
"""
E2SFCA Healthcare Facility Location - Brazil
Based on Leitch & Wei (2024), Annals of Operations Research

CORREÇÕES APLICADAS:
1. Equidade (Z): Só aplicada a nós que REALMENTE podem ser alcançados por algum candidato.
2. Capacidade diferenciada: R_j = x_j / DEMANDA_PONDERADA_j é calculado apenas
   sobre a demanda DENTRO da catchment area de j, então cada localização tem um peso
   diferente — o solver passa a distinguir localizações de verdade.
3. Sem variável R[] no Gurobi: Substituímos R_j diretamente na expressão de A_i,
   tornando o modelo menor e mais claro.
"""

import gurobipy as gp
from gurobipy import GRB, quicksum
import pandas as pd
import numpy as np

# =============================================================================
# 1. CARREGAR DADOS (ajustar os caminhos conforme a localização dos dados)
# =============================================================================
origem_alvo_df = pd.read_csv('Documents/SP_coords_medias_divisoes.csv')
facilidades_df = pd.read_csv('Hospitais_gerais_SP_CNES.csv')

# Matriz de tempos: linhas = nós de demanda, colunas = D_xxx (candidatos) + F_xxx (existentes)
A_total_df = pd.read_csv('matriz_distancia_SP_haversine.csv')
A_total_df = A_total_df.fillna(1e9)  # Pares sem rota -> tempo infinito

col_dem = [c for c in A_total_df.columns if c.startswith('D_')]
col_fac = [c for c in A_total_df.columns if c.startswith('F_')]
colunas  = col_dem + col_fac

A_total_df.index = col_dem
durations_min = A_total_df[colunas].to_numpy()

# Demanda d_i
demanda_df = pd.read_csv('setores_populacao_ajustada_35.csv').rename(columns={"CD_setor": "CD_SETOR"})
setores_df = pd.read_csv('SP_setores_populacao.csv', sep=';')
df_join = demanda_df.merge(setores_df, on="CD_SETOR", how="right").fillna(0)
agrupado = df_join.groupby("ID_divisao")["necessidade_total_leitos"].sum()
d_i = {f"D_{k}": v for k, v in agrupado.to_dict().items()}

# Suprimentos existentes
x_hat = {f"F_{row['CO_UNIDADE']}": row['leitos'] for _, row in facilidades_df.iterrows()}

In [2]:
# =============================================================================
# 2. CONJUNTOS E PARÂMETROS
# =============================================================================
t_0   = 45    # Tempo máximo aceitável de viagem (minutos)
P_max = 12     # Número de novas facilidades a abrir
Q_max = 1200   # Total de leitos novos a distribuir
k_min = 50    # Mínimo de leitos por nova facilidade aberta
k_max = 200   # Máximo de leitos por nova facilidade aberta
alpha = 1   # 0 = só equidade, 1 = só eficiência, 0.5 = balanceado

N   = col_dem           # Nós de demanda
M_e = col_fac           # Facilidades existentes
M_n = col_dem           # Candidatos a novas facilidades (mesmos nós de demanda)
M   = M_n + M_e         # Todas as facilidades

# Áreas de captação (catchment): M_i[i] = facilidades alcançáveis a partir de i
M_i = {
    col_dem[i]: [colunas[j] for j in range(durations_min.shape[1])
                 if durations_min[i, j] <= t_0]
    for i in range(len(col_dem))
}

# N_j[j] = nós de demanda alcançados pela facilidade j
N_j = {
    colunas[j]: [col_dem[i] for i in range(len(col_dem))
                 if durations_min[i, j] <= t_0]
    for j in range(durations_min.shape[1])
}

# Função de decaimento w_ij (exponencial negativa, β=0.09 como no artigo)
beta = 0.09
w_df = pd.DataFrame(
    np.exp(-beta * durations_min),
    index=col_dem,
    columns=colunas
)

In [3]:
# =============================================================================
# 3. PRÉ-CÁLCULO: DEMANDA PONDERADA POR FACILIDADE
# =============================================================================
EPSILON = 1e-6

# Denominador padrão (E2SFCA clássico): usado para facilidades EXISTENTES
# e para calcular A_antes (acessibilidade base).
DEMANDA_PONDERADA = {}
for j in colunas:
    dem_total = sum(w_df.loc[i, j] * d_i.get(i, 0.0) for i in N_j[j])
    DEMANDA_PONDERADA[j] = max(dem_total, EPSILON)

In [4]:
# =============================================================================
# 4. ACESSIBILIDADE ANTES DA OTIMIZAÇÃO (baseline, só facilidades existentes)
# =============================================================================
R_existente = {j: x_hat[j] / DEMANDA_PONDERADA[j] for j in M_e}

A_antes = {}
for i in N:
    soma = sum(w_df.loc[i, j] * R_existente[j]
               for j in M_i[i] if j in M_e)
    A_antes[i] = soma

# =============================================================================
# PRÉ-CÁLCULO: Demanda não-atendida (SHORTAGE), usado no OBJETIVO
#
# A solução correta é manter o denominador E2SFCA padrão (d_i puro),
# e usar d_shortage = d_i * shortage_i como peso no OBJETIVO.
# Isso faz coeff[j] = Σ_{i∈N_j} d_shortage_i * w_ij / DEMANDA_PONDERADA[j],
# que é alto onde há carência real e baixo onde já há boa cobertura.
# =============================================================================
A_vals_pos = [v for v in A_antes.values() if v > 0]
limiar_acesso = np.median(A_vals_pos) if A_vals_pos else EPSILON
print(f"Limiar de acesso adequado (mediana A_antes > 0): {limiar_acesso:.4f}")

shortage   = {i: max(1.0 - A_antes[i] / limiar_acesso, 0.0) for i in N}
d_shortage = {i: d_i.get(i, 0.0) * shortage[i] for i in N}

DEMANDA_PONDERADA_CAND = {}
for j in M_n:
    dem_total = sum(w_df.loc[i, j] * d_shortage.get(i, 0.0) for i in N_j[j])
    DEMANDA_PONDERADA_CAND[j] = max(dem_total, EPSILON)

Limiar de acesso adequado (mediana A_antes > 0): 1.6028


In [5]:
# =============================================================================
# 5. FILTRO DE EQUIDADE
#
# CORREÇÃO DO BUG PRINCIPAL:
# Só aplicamos a restrição Z <= d_i * A_i para nós i que:
#   (a) têm demanda > 0, E
#   (b) podem ser ALCANÇADOS por pelo menos um CANDIDATO (M_n)
#
# Nós sem candidatos ao alcance nunca terão A_i melhorada, então forçá-los
# a satisfazer Z <= d_i*A_i puxaria Z para zero inutilmente.
# =============================================================================
nos_com_candidato = {
    i for i in N
    if d_i.get(i, 0.0) > 0.5
    and any(j in M_n for j in M_i.get(i, []))
}

print(f"Total de nós: {len(N)}")
print(f"Nós com candidato alcançável e demanda > 0: {len(nos_com_candidato)}")
print(f"Nós excluídos da restrição de equidade: {len(N) - len(nos_com_candidato)}")

Total de nós: 3146
Nós com candidato alcançável e demanda > 0: 2566
Nós excluídos da restrição de equidade: 580


In [6]:
# =============================================================================
# NORMALIZAÇÃO PELO PONTO UTOPIA
# Resolve o modelo com alpha=1 e alpha=0 para obter os valores máximos
# de cada objetivo separadamente. Isso torna a combinação convexa interpretável:
# um alpha=0.5 real significa 50% do peso em cada objetivo, não 50% de magnitudes
# arbitrárias.
# =============================================================================

def resolver_modelo(alpha_val, N, M_n, M_e, M_i, d_i, d_shortage,
                    DEMANDA_PONDERADA, x_hat, w_df,
                    P_max, Q_max, k_min, k_max, nos_com_candidato,
                    silencioso=True):
    """
    Constrói e resolve o modelo para um dado alpha.
    Retorna (valor_eficiencia, valor_Z, status).
    valor_eficiencia = (1/|N|) * Σ d_shortage_i * A_i
    valor_Z          = min_{i ∈ N*} d_i * A_i
    """
    mod = gp.Model()
    if silencioso:
        mod.setParam('OutputFlag', 0)
    mod.setParam('TimeLimit', 300)
    mod.setParam('MIPGap', 0.001)

    y = mod.addVars(M_n, vtype=GRB.BINARY)
    x = mod.addVars(M_n, vtype=GRB.INTEGER)
    Z = mod.addVar(vtype=GRB.CONTINUOUS)
    mod.update()

    # Mesma expr_A_i, mas local a esta função
    def _expr_A(i):
        e = gp.LinExpr()
        for j in M_i.get(i, []):
            w = w_df.loc[i, j]
            denom = DEMANDA_PONDERADA[j]
            if j in M_n:
                e.add(x[j], w / denom)
            elif j in M_e:
                e.addConstant(w * x_hat[j] / denom)
        return e

    ef = (1.0 / len(N)) * quicksum(d_shortage[i] * _expr_A(i) for i in N)
    mod.setObjective(alpha_val * ef + (1 - alpha_val) * Z, GRB.MAXIMIZE)

    for i in nos_com_candidato:
        mod.addConstr(d_i[i] * _expr_A(i) >= Z)
    for j in M_n:
        mod.addConstr(x[j] >= k_min * y[j])
        mod.addConstr(x[j] <= k_max * y[j])
    mod.addConstr(quicksum(y[j] for j in M_n) == P_max)
    mod.addConstr(quicksum(x[j] for j in M_n) == Q_max)

    mod.optimize()

    if mod.SolCount > 0:
        # Calcular os dois componentes separadamente na solução encontrada
        ef_val = (1.0 / len(N)) * sum(
            d_shortage[i] * sum(
                w_df.loc[i, j] * (x[j].x if j in M_n else x_hat.get(j, 0))
                / DEMANDA_PONDERADA[j]
                for j in M_i.get(i, [])
            )
            for i in N
        )
        Z_val = Z.x
        return ef_val, Z_val, mod.status
    else:
        return None, None, mod.status

In [7]:
print("\nCalculando ponto utopia (2 resoluções auxiliares)...")

Ef_star, _, _ = resolver_modelo(
    1.0, N, M_n, M_e, M_i, d_i, d_shortage,
    DEMANDA_PONDERADA, x_hat, w_df,
    P_max, Q_max, k_min, k_max, nos_com_candidato
)

_, Z_star, _ = resolver_modelo(
    0.0, N, M_n, M_e, M_i, d_i, d_shortage,
    DEMANDA_PONDERADA, x_hat, w_df,
    P_max, Q_max, k_min, k_max, nos_com_candidato
)

# Proteção contra casos degenerados
Ef_star = Ef_star if (Ef_star and Ef_star > EPSILON) else 1.0
Z_star  = Z_star  if (Z_star  and Z_star  > EPSILON) else 1.0

print(f"  Eficiência máxima (alpha=1): {Ef_star:.6f}")
print(f"  Equidade máxima  (alpha=0): {Z_star:.6f}")


Calculando ponto utopia (2 resoluções auxiliares)...
Set parameter Username
Set parameter LicenseID to value 2780008
Academic license - for non-commercial use only - expires 2027-02-17
  Eficiência máxima (alpha=1): 1.130236
  Equidade máxima  (alpha=0): 0.082216


In [8]:
# =============================================================================
# 6. MODELO GUROBI
# =============================================================================
m = gp.Model("E2SFCA_Sergipe")

# Variáveis de decisão
y = m.addVars(M_n, vtype=GRB.BINARY,   name="y")   # 1 se j é aberta
x = m.addVars(M_n, vtype=GRB.INTEGER,  name="x")   # leitos alocados a j
Z = m.addVar(        vtype=GRB.CONTINUOUS, name="Z") # Pior acessibilidade ponderada
m.update()

# =============================================================================
# EXPRESSÃO DE ACESSIBILIDADE A_i (sem variável auxiliar no Gurobi)
#
# A_i = Σ_{j ∈ M_i(i)} w_ij * R_j
#
# Para facilidades existentes: R_j = x_hat[j] / DEMANDA_PONDERADA[j]  (constante)
# Para candidatas novas:       R_j = x[j] / DEMANDA_PONDERADA[j]      (variável)
#
# Portanto A_i é uma expressão linear em x[j].
#
# NOTA: Esta substituição direta elimina a necessidade das variáveis R[] e A[]
# no Gurobi, tornando o modelo mais limpo e sem bugs de linkagem.
# =============================================================================
def expr_A_i(i):
    expr = gp.LinExpr()
    for j in M_i.get(i, []):
        w = w_df.loc[i, j]
        denom = DEMANDA_PONDERADA[j]   # padrão para todos
        if j in M_n:
            expr.add(x[j], w / denom)
        elif j in M_e:
            expr.addConstant(w * x_hat[j] / denom)
    return expr

# =============================================================================
# FUNÇÃO OBJETIVO (Eq. 4a do artigo, com denominador corrigido)
#
# max α * (1/|N|) * Σ_i d_i * A_i   +   (1-α) * Z
#
# Com DEMANDA_PONDERADA_CAND no denominador de R_j para candidatos,
# os coeficientes de x[j] no objetivo passam a ser:
#   coeff[j] = Σ_{i∈N_j} d_i * w_ij / DEMANDA_PONDERADA_CAND[j]
# Como DEMANDA_PONDERADA_CAND usa d_shortage ≠ d_i, o numerador e denominador
# diferem → coeff[j] ≠ 1 → o objetivo varia com a localização. Resolvido.
# =============================================================================
eficiencia_norm = (1.0 / (len(N) * Ef_star)) * quicksum(d_shortage[i] * expr_A_i(i) for i in N)
equidade_norm   = (1.0 / Z_star) * Z
m.setObjective(alpha * eficiencia_norm + (1 - alpha) * equidade_norm, GRB.MAXIMIZE)

# =============================================================================
# RESTRIÇÕES
# =============================================================================

# C1: Equidade — Z <= d_i * A_i, apenas para nós alcançáveis por candidatos
for i in nos_com_candidato:
    m.addConstr(d_i[i] * expr_A_i(i) >= Z, name=f"equity_{i}")

# C2: Capacidade das novas facilidades (k_min <= x_j <= k_max se aberta, 0 se fechada)
for j in M_n:
    m.addConstr(x[j] >= k_min * y[j], name=f"cap_min_{j}")
    m.addConstr(x[j] <= k_max * y[j], name=f"cap_max_{j}")

# C3: Número exato de novas facilidades
m.addConstr(quicksum(y[j] for j in M_n) == P_max, name="num_facilities")

# C4: Total de leitos distribuídos
m.addConstr(quicksum(x[j] for j in M_n) == Q_max, name="total_beds")

m.update()
print(f"\nModelo: {m.numVars} variáveis, {m.numConstrs} restrições")

# =============================================================================
# 7. OTIMIZAÇÃO
# =============================================================================
m.setParam('TimeLimit', 600)  # 10 minutos
m.setParam('MIPGap', 0.001)   # Para quando gap <= 0.1%
m.optimize()

# =============================================================================
# 8. RESULTADOS
# =============================================================================
if m.status in (GRB.OPTIMAL, GRB.TIME_LIMIT) and m.SolCount > 0:
    abertas = [j for j in M_n if y[j].x > 0.5]

    # Calcular A_depois com os valores ótimos
    A_depois = {}
    for i in N:
        soma = 0.0
        for j in M_i.get(i, []):
            w = w_df.loc[i, j]
            denom = DEMANDA_PONDERADA[j]
            if j in M_n:
                soma += w * x[j].x / denom
            elif j in M_e:
                soma += w * x_hat[j] / denom
        A_depois[i] = soma
    
    # Cálculo dos componentes normalizados
    ef_val = (1.0 / len(N)) * sum(
    d_shortage[i] * sum(
        w_df.loc[i, j] * (x[j].x if j in M_n else x_hat.get(j, 0))
        / DEMANDA_PONDERADA[j]
        for j in M_i.get(i, [])) for i in N)

    print(f"Função objetivo normalizada:  {m.objVal:.6f}  (em [0,1], idealmente)")
    print(f"  Eficiência: {ef_val:.6f} / {Ef_star:.6f} = {ef_val/Ef_star:.4f}")
    print(f"  Equidade:   {Z.x:.6f}  / {Z_star:.6f}  = {Z.x/Z_star:.4f}")

    print(f"\n{'='*60}")
    print(f"RESULTADOS (α={alpha})")
    print(f"{'='*60}")
    print(f"Objetivo: {m.objVal:.6f}")
    print(f"Z (equidade, pior d_i*A_i entre nós alcançáveis): {Z.x:.6f}")

    media_antes  = np.mean(list(A_antes.values()))
    media_depois = np.mean(list(A_depois.values()))
    print(f"Média A_i antes: {media_antes:.4f} | depois: {media_depois:.4f}")

    print(f"\nFACILIDADES ABERTAS ({len(abertas)}):")
    for j in abertas:
        leitos = x[j].x
        denom  = DEMANDA_PONDERADA[j]
        R_j    = leitos / denom
        cobertos = N_j.get(j, [])
        print(f"  {j}: {leitos:.0f} leitos | R_j={R_j:.4f} | "
              f"{len(cobertos)} nós cobertos | dem.ponderada={denom:.2f}")

    # Exportar para QGIS
    df_out = pd.DataFrame({
        "ID_modelo": list(N),
        "d_i":       [d_i.get(i, 0) for i in N],
        "A_antes":   [A_antes[i] for i in N],
        "A_depois":  [A_depois[i] for i in N],
        "A_ganho":   [A_depois[i] - A_antes[i] for i in N],
        "dA_antes":  [d_i.get(i,0) * A_antes[i]  for i in N],
        "dA_depois": [d_i.get(i,0) * A_depois[i] for i in N],
    })
    df_out["ID_qgis"] = df_out["ID_modelo"].str.replace("D_", "")
    df_out.to_csv(f"acessibilidade_alpha{alpha:.1f}.csv", index=False)

    df_fac = pd.DataFrame({
        "ID_modelo":       abertas,
        "leitos":          [x[j].x for j in abertas],
        "dem_ponderada":   [DEMANDA_PONDERADA.get(j, 0) for j in abertas],
        "R_j":             [x[j].x / DEMANDA_PONDERADA.get(j, EPSILON) for j in abertas],
    })
    df_fac["ID_qgis"] = df_fac["ID_modelo"].str.replace("D_", "")
    df_fac.to_csv(f"facilidades_alpha{alpha:.1f}.csv", index=False)

    print(f"\nArquivos exportados: acessibilidade_alpha{alpha:.1f}.csv e facilidades_alpha{alpha:.1f}.csv")

else:
    print(f"Modelo sem solução. Status: {m.status}")


Modelo: 6293 variáveis, 8860 restrições
Set parameter TimeLimit to value 600
Set parameter MIPGap to value 0.001
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 5 5500U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
TimeLimit  600
MIPGap  0.001

Optimize a model with 8860 rows, 6293 columns and 475597 nonzeros
Model fingerprint: 0xa1e9dcb1
Variable types: 1 continuous, 6292 integer (3146 binary)
Coefficient statistics:
  Matrix range     [5e-06, 2e+02]
  Objective range  [4e-07, 3e-04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e-02, 2e+03]
Found heuristic solution: objective 0.7102118
Presolve removed 2566 rows and 1 columns
Presolve time: 0.06s
Presolved: 6294 rows, 6292 columns, 18876 nonzeros
Variable types: 0 continuous, 6292 integer (3146 binary)

Root relaxation: objective 1.000000e+00, 5129 iterations